In [1]:
import numpy as np
import tensorflow as tf
import cv2
from tensorflow.keras.layers import Input,Conv2D,Conv2DTranspose,Flatten,Dense,MaxPooling2D
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy, MeanSquaredError
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
batchSize = 16
imageSize = 128
epochs = 200

In [3]:
import kagglehub
path = kagglehub.dataset_download("shravankumar9892/image-colorization")
print(path)

C:\Users\morga\.cache\kagglehub\datasets\shravankumar9892\image-colorization\versions\4


In [4]:
grayPath = f"{path}/l/gray_scale.npy"
for i in range(1):
    colPath = f"{path}/ab/ab/ab{i+1}.npy"
colImage = np.load(colPath)[:100]
grayImage = np.load(grayPath)[:100]

In [5]:
print(colImage.shape)
print(grayImage.shape)

(100, 224, 224, 2)
(100, 224, 224)


In [12]:
def preprocessing(grayImage,colImage):
    grayResize = np.array([cv2.resize(image,(imageSize,imageSize))for image in grayImage])
    colResize = np.array([cv2.resize(image,(imageSize,imageSize))for image in colImage])
    grayResize = grayResize[...,np.newaxis]
    colResize = colResize/255
    if colResize.shape[-1] == 1:
        print("colImage has 1 channel - expanding it to 3")
        colResize = np.repeat(colResize,3,axis=-1)

    elif colResize.shape[-1] == 2:
        print("colImage has 2 channels - expanding to 3")
        colResize = np.concatenate((colResize,colResize[...,:1]),axis=-1)

    elif colResize.shape[-1] != 3:
        print("unexpected colImage with more than 3 channels")
        raise ValueError(f"ERROR - unexpected image with {colResize.shape[-1]} channels")

    return grayResize.shape, colResize.shape

gImage, cImage = preprocessing(grayImage,colImage)

colImage has 2 channels - expanding to 3


In [13]:
Xtrain,Xtest,ytrain,ytest = train_test_split(gImage,cImage,test_size=0.2,random_state=1)
trainDS = tf.data.Dataset.from_tensor_slices((Xtrain,ytrain)).batch(batchSize)
testDS  = tf.data.Dataset.from_tensor_slices((Xtest,ytest)).batch(batchSize)

In [ ]:
from tensorflow.keras.layers import LeakyReLU
def generate():
    inputs = Input(shape=(imageSize,imageSize,1))
    X = Conv2D(16,(5,5),padding="same")